In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
list_files = os.listdir("../")
list_files

['.git',
 'away_team.csv',
 'away_team_score.csv',
 'event.csv',
 'home_team.csv',
 'home_team_score.csv',
 'Javadi',
 'notebook.ipynb',
 'odds.csv',
 'pbp.csv',
 'power.csv',
 'round.csv',
 'season.csv',
 'statistics.csv',
 'time.csv',
 'tournament.csv',
 'venue.csv',
 'votes.csv']

In [3]:
df_pbp = pd.read_csv("../pbp.csv")
event_df = pd.read_csv("../event.csv")

In [14]:
df_pbp

,match_id,set_id,game_id,point_id,home_point,away_point,point_description,home_point_type,away_point_type,home_score,away_score,serving,scoring
0,11998445,3,13,0,1,0,0,6,5,6,7,1,2
1,11998445,3,13,1,1,1,0,5,6,6,7,1,2
2,11998445,3,13,2,1,2,0,5,6,6,7,1,2
3,11998445,3,13,3,1,3,0,5,1,6,7,1,2
4,11998445,3,13,4,1,4,0,5,1,6,7,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2549364,12213803,1,1,4,30,40,0,1,2,1,0,1,1
2549365,12213803,1,1,5,40,40,0,1,5,1,0,1,1
2549366,12213803,1,1,6,A,40,0,1,5,1,0,1,1
2549367,12213803,1,1,7,40,40,0,5,1,1,0,1,1


In [4]:
df_statistics = pd.read_csv("../statistics.csv")

In [ ]:
# THRESHOLDS
# Professional tennis: minimum 2 sets, maximum 5 sets per match
# set_id must be between 1 and 5

MIN_SETS = 2
MAX_SETS = 5
MIN_SET_ID = 1
MAX_SET_ID = 5

report = []
total_original = len(df_pbp)

def log(step, desc, removed, note=''):
    report.append({
        'Step' : step,
        'Description' : desc,
        'Rows Removed' : removed,
        '% of Original' : round(removed / total_original * 100, 2),
        'Note' : note })

# DIAGNOSTIC — before any cleaning, check raw set_id distribution

print("RAW DIAGNOSTICS — df_pbp")
print(f"Total rows : {len(df_pbp):,}")
print(f"Unique match_ids : {df_pbp['match_id'].nunique():,}")
print(f"\nset_id distribution (raw):")
print(df_pbp['set_id'].value_counts().sort_index().to_string())
print(f"\nset_id null count : {df_pbp['set_id'].isna().sum():,}")
print(f"set_id min : {df_pbp['set_id'].min()}")
print(f"set_id max : {df_pbp['set_id'].max()}")

# STEP 1: Drop exact duplicate rows

df = df_pbp.copy()
before = len(df)
df = df.drop_duplicates()
removed = before - len(df)
log(1, 'Exact duplicate rows', removed, '')

# STEP 2: Drop rows with null match_id or null set_id
# These are completely unidentifiable rows

before = len(df)
df = df[df['match_id'].notna() & df['set_id'].notna()].copy()
removed = before - len(df)
log(2, 'Rows with null match_id or null set_id', removed, '')

# STEP 3: Drop rows with invalid set_id values
# set_id must be 1–5 in professional tennis

before = len(df)
invalid_sets = (df['set_id'] < MIN_SET_ID) | (df['set_id'] > MAX_SET_ID)
df = df[~invalid_sets].copy()
removed = before - len(df)
log(3, f'Rows with set_id outside valid range ({MIN_SET_ID}–{MAX_SET_ID})', removed, '')

# STEP 4: Cross-validate with event_df — only keep finished matches

before = len(df)
valid_ids = event_df[event_df['winner_code'].isin([1, 2])]['match_id']
df = df[df['match_id'].isin(valid_ids)].copy()
removed = before - len(df)
log(4, 'Rows from unfinished matches (null/invalid winner_code)', removed, '')


# STEP 5: Handle duplicate match_ids — corrupt vs interrupted
# Here we work at the MATCH level, not the row level
# A match is "duplicated" if the same match_id has suspiciously
# overlapping point sequences, suggesting the same data was recorded twice


# First, get match-level summary — sets per match
match_sets = (
    df.groupby('match_id')['set_id']
    .agg(
        unique_sets = 'nunique',
        max_set = 'max',
        min_set = 'min',
        total_points = 'count' )
    .reset_index())

# Flag matches where max_set != unique_sets
# e.g. set_ids are [1, 1, 2, 2] → unique=2, max=2 → fine
# e.g. set_ids are [1, 3] → unique=2, max=3 → gap in set sequence → corrupt
has_gap = match_sets['max_set'] != match_sets['unique_sets']
gap_ids = set(match_sets[has_gap]['match_id'])

before = len(df)
df = df[~df['match_id'].isin(gap_ids)].copy()
removed = before - len(df)
log(5, 'Matches with gaps in set_id sequence (e.g. sets 1,3 but no 2)',
    removed, 'Indicates missing or corrupt point data')


# STEP 6: Recalculate sets per match after cleaning

sets_per_match = (
    df.groupby('match_id')['set_id']
    .nunique()
    .reset_index()
    .rename(columns={'set_id': 'sets_played'}))


# STEP 7: Remove matches with fewer than 2 sets (retirements/incomplete)

before = sets_per_match['sets_played'].lt(MIN_SETS).sum()
sets_per_match = sets_per_match[sets_per_match['sets_played'] >= MIN_SETS].copy()
removed = before
log(7, f'Matches with fewer than {MIN_SETS} sets (retirements/walkovers)', removed, '')


# STEP 8: Remove matches with more than 5 sets (physically impossible)

before = (sets_per_match['sets_played'] > MAX_SETS).sum()
sets_per_match = sets_per_match[sets_per_match['sets_played'] <= MAX_SETS].copy()
removed = before
log(8, f'Matches with more than {MAX_SETS} sets (impossible)', removed, '')


# STEP 9: Cross-validate against df_statistics
# If statistics says 3 periods but pbp only shows 2 sets, flag it
# We keep the pbp result but log the discrepancies

stats_sets = (
    df_statistics[df_statistics['period'] != 'ALL']
    .groupby('match_id')['period']
    .nunique()
    .reset_index()
    .rename(columns={'period': 'stats_sets'}))

comparison = sets_per_match.merge(stats_sets, on='match_id', how='inner')
discrepancy = comparison[
    comparison['sets_played'] != comparison['stats_sets']
]
n_discrepancy = len(discrepancy)

report.append({
    'Step'          : 9,
    'Description'   : 'Matches where pbp set count disagrees with statistics periods',
    'Rows Removed'  : f'{n_discrepancy} flagged (kept — pbp is ground truth)',
    '% of Original' : '-',
    'Note'          : 'Use pbp count as authoritative source'
})

print(f"\n  Cross-validation: {n_discrepancy} matches disagree "
      f"between pbp and statistics set counts")


# DESCRIPTIVE STATISTICS

sets   = sets_per_match['sets_played']
desc   = sets.describe(percentiles=[0.25, 0.5, 0.75])
vc     = sets.value_counts().sort_index()
vc_pct = (sets.value_counts(normalize=True).sort_index() * 100).round(2)



RAW DIAGNOSTICS — df_pbp
Total rows          : 2,549,369
Unique match_ids    : 10,956

set_id distribution (raw):
set_id
1    1132664
2    1103250
3     313455

set_id null count   : 0
set_id min          : 1
set_id max          : 3

  Cross-validation: 9 matches disagree between pbp and statistics set counts


In [8]:
total_clean   = len(sets_per_match)
total_removed = df_pbp['match_id'].nunique() - total_clean

print("DATA CLEANING REPORT — Sets Played (from df_pbp)")
print(f"  Original unique matches : {df_pbp['match_id'].nunique():,}")
print(f"  Clean matches           : {total_clean:,}")
print(f"  Matches removed         : {total_removed:,}")
print(f"  Clean data              : {round(total_clean/df_pbp['match_id'].nunique()*100,2)}%")
print(pd.DataFrame(report).to_string(index=False))



DATA CLEANING REPORT — Sets Played (from df_pbp)
  Original unique matches : 10,956
  Clean matches           : 10,870
  Matches removed         : 86
  Clean data              : 99.22%
 Step                                                   Description                           Rows Removed % of Original                                    Note
    1                                          Exact duplicate rows                                1294625         50.78                                        
    2                        Rows with null match_id or null set_id                                      0           0.0                                        
    3                    Rows with set_id outside valid range (1–5)                                      0           0.0                                        
    4       Rows from unfinished matches (null/invalid winner_code)                                     79           0.0                                        
    5 Matc

In [9]:
print("DESCRIPTIVE STATISTICS — Sets Played per Match")
print(f"  Count   : {int(desc['count']):,}")
print(f"  Mean    : {desc['mean']:.4f}")
print(f"  Std Dev : {desc['std']:.4f}")
print(f"  Min     : {int(desc['min'])}")
print(f"  25%     : {desc['25%']:.1f}")
print(f"  Median  : {desc['50%']:.1f}")
print(f"  75%     : {desc['75%']:.1f}")
print(f"  Max     : {int(desc['max'])}")

print("\nDISTRIBUTION:\n")
print(f"  {'Sets':>5}  {'Matches':>8}  {'%':>7}  Bar")
print(f"  {'-'*5}  {'-'*8}  {'-'*7}  {'-'*30}")
for n in vc.index:
    bar = '█' * int(vc_pct[n] / 2)
    print(f"  {n:>5}  {vc[n]:>8,}  {vc_pct[n]:>6.2f}%  {bar}")

DESCRIPTIVE STATISTICS — Sets Played per Match
  Count   : 10,870
  Mean    : 2.3048
  Std Dev : 0.4603
  Min     : 2
  25%     : 2.0
  Median  : 2.0
  75%     : 3.0
  Max     : 3

DISTRIBUTION:

   Sets   Matches        %  Bar
  -----  --------  -------  ------------------------------
      2     7,557   69.52%  ██████████████████████████████████
      3     3,313   30.48%  ███████████████
